In [ ]:
import rasterio
import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


def get_metrics(y_pred, y_test):
    print(
        f"""
    R2: {round(r2_score(y_test, y_pred), 2)}
    MSE: {round(mean_squared_error(y_test, y_pred), 2)}
    MAE: {round(mean_absolute_error(y_test, y_pred), 2)}
        """
    )

    plt.hist(y_test, label='test', alpha=0.5, bins=100)
    plt.hist(y_pred, label='pred', alpha=0.5, bins=100)
    plt.xlabel('Volume, m3/ha')
    plt.ylabel('count')
    plt.legend()

### Read data

In [ ]:
y_src = rasterio.open('data/volume.tiff')

volume = y_src.read(1)
volume = np.where(volume < 0, 0, volume)

In [ ]:
plt.imshow(volume)
plt.colorbar()

In [ ]:
x_src = rasterio.open('data/source_11647_2024.tiff')

In [ ]:
bands = x_src.read()
bands.shape

### Preprocessing

In [ ]:
volume = np.expand_dims(volume, axis=0)
data = np.concatenate( (bands, volume), axis=0)

In [ ]:
df = data.reshape([15, data.shape[1] * data.shape[2]])

In [ ]:
df = pd.DataFrame(
    df.T,
    columns=["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09", "B11","B12","SCL","CLM", "volume"])
df = df.replace(65535.0, np.nan)
df.dropna(inplace=True)

In [ ]:
features = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
       'B11', 'B12', 'SCL', 'CLM']

scaler = StandardScaler()
data = scaler.fit_transform(df[features].copy())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data, df.volume, test_size=0.33, random_state=42)

### Models

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X_train, y_train)

In [ ]:
y_pred = ridge_model.predict(X_test)
y_pred = np.where(y_pred < 0, 0, y_pred)

In [ ]:
get_metrics(y_pred, y_test)